In [ ]:
# Locate the repository when opened from any experiment subdirectory.
import os
import sys
from pathlib import Path

_here = Path.cwd().resolve()
ROOT = next((p for p in (_here, *_here.parents)
             if (p / "smoke_test.py").is_file()
             and (p / "configs/config.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the PGAE-tau repository.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print(f"Repository: {ROOT}")


In [ ]:
import matplotlib.pyplot as plt
from configs.config import sys_config
from configs.nn_setting import nn_setting
from tqdm import tqdm
from matplotlib import cm

In [ ]:
import torch
import sys

from data.loaders import load_case, load_measurement, load_load_pv, load_dataset

from models.baselines.graph_ae import GraphAE, BusPQDataset

from data.dataset import rnn_dataset

from torch.utils.data import DataLoader
import numpy as np
from torch import optim
from data.dataset import scaler
from time import time

In [ ]:
class Evaluation:
    """
    GraphAE evaluation with case-configured ChebConv layers.
    ==========================================================
    Input: per-bus [P_inj, Q_inj] from voltage + Ybus.
    Threshold calibrated on validation set BusPQDataset.
    Author: Hawkyu | 2026-06-19
    """
    def __init__(self, case_class):

        self.nn_setting = nn_setting
        self.case_class = case_class
        self.case_class.gen_torch_constant()

        # ---- Load GraphAE model ----
        model = GraphAE().to(nn_setting['device'])
        model_path = f"saved_model/{sys_config['case_name']}/GraphAE.pt"

        model.load_state_dict(torch.load(model_path,
            map_location=torch.device(nn_setting['device']), weights_only=True))

        device = nn_setting['device']
        print(f'Using {device} device')
        model.to(device)
        model.eval()
        print(model)

        self.model = model
        self.device = device

        # ---- Ybus for P/Q computation ----
        self.Ybus = torch.tensor(case_class.Ybus, dtype=torch.cfloat, device=device)

        # ---- Validation threshold via BusPQDataset ----
        valid_set = BusPQDataset(mode='valid')
        self.f_min = valid_set.f_min.to(device)
        self.f_max = valid_set.f_max.to(device)
        self.f_diff = self.f_max - self.f_min + 1e-7

        valid_loader = DataLoader(valid_set, batch_size=1, shuffle=False)

        loss_fn = torch.nn.MSELoss()
        valid_loss = []
        with torch.no_grad():
            for inp, tgt in valid_loader:
                inp = inp.to(device)
                tgt = tgt.to(device)
                recon, _ = model(inp)
                recon_norm = (recon - self.f_min) / self.f_diff
                tgt_norm   = (tgt   - self.f_min) / self.f_diff
                loss = loss_fn(recon_norm, tgt_norm)
                valid_loss.append(loss.item())

        # ---- Thresholds ----
        quantile = [0.80, 0.82, 0.84, 0.86, 0.88, 0.90, 0.92, 0.94, 0.96, 0.98, 0.999]
        ae_threshold = np.quantile(valid_loss, quantile)
        self.ae_threshold = ae_threshold
        self.quantile_idx = 6       # 0.92 quantile
        self.quantile = quantile

        print(f'Detection threshold: {self.ae_threshold}')

    def compute_pq(self, v_est):
        """
        Compute per-bus [P, Q] from complex voltage v_est [N].
        S = V ⊙ conj(Ybus @ V)  →  P = real(S), Q = imag(S).
        """
        if not torch.is_tensor(v_est):
            v_est = torch.from_numpy(v_est)
        v = v_est.cfloat().to(self.device)
        I = torch.matmul(self.Ybus, v)
        S = v * torch.conj(I)
        pq = torch.stack([S.real.float(), S.imag.float()], dim=-1)  # [N, 2]
        return pq

    def evaluate(self, pq_features):
        """
        Evaluate reconstruction error for per-bus [P, Q] features.
        pq_features: [N, 2] normalized tensor.
        Returns: (recon_numpy, loss_recons).
        """
        if not torch.is_tensor(pq_features):
            pq_features = torch.tensor(pq_features, dtype=torch.float32)
        pq_features = pq_features.float().to(self.device).unsqueeze(0)  # [1, N, 2]

        loss_fn = torch.nn.MSELoss()
        self.model.eval()
        with torch.no_grad():
            recon, _ = self.model(pq_features)
            recon_norm = (recon - self.f_min) / self.f_diff
            tgt_norm = (pq_features - self.f_min) / self.f_diff
            loss_recons = loss_fn(recon_norm, tgt_norm)

        return recon.squeeze(0).cpu().numpy(), loss_recons.item()

In [ ]:
"""
Test the performance of attack detection and identification algorithms
======================================================================
GraphAE: per-bus P/Q input and case-configured ChebConv layers.
Author: Hawkyu | 2026-06-19
"""

from configs.config import sys_config, save_metric
from data.loaders import load_case, load_measurement, load_load_pv

# Load cases, measurement, and load
case_class = load_case()
z_noise_summary, v_est_summary = load_measurement()
load_active, load_reactive, pv_active_, pv_reactive_ = load_load_pv()
test_dataloader_scaled, test_dataloader_unscaled, valid_dataloader_scaled, valid_dataloader_unscaled = load_dataset()


dd_detector = Evaluation(case_class=case_class)
scaler_ = scaler()
print(f'Quantile: {dd_detector.quantile[dd_detector.quantile_idx]}')
print(f'Threshold: {dd_detector.ae_threshold[dd_detector.quantile_idx]}')

# ---- FP_DDD & TP_DDD ----
# Attack list
ang_no_list = [1,2,3]
mag_no = 0
ang_str_list = [0.30]
mag_str = 0

# FP_DDD: false positives on normal test samples
FP_DDD = []
test_pq_set = BusPQDataset(mode='test')
print(f'Test P/Q samples: {len(test_pq_set)}')

for i in range(len(test_pq_set)):
    pq, _ = test_pq_set[i]
    _, loss_recons = dd_detector.evaluate(pq)
    if loss_recons <= dd_detector.ae_threshold[dd_detector.quantile_idx]:
        FP_DDD.append(False)
    else:
        FP_DDD.append(True)

# TP_DDD: true positives on attacked test samples
TP_DDD = {}
att_strength = {}
print('Computing TPR of GraphAE...')

for ang_no in ang_no_list:
    for ang_str in ang_str_list:
        print(f'Attack: ang_no={ang_no}, ang_str={ang_str}')
        TP_DDD[f'({ang_no},{ang_str})'] = []
        att_strength[f'({ang_no},{ang_str})'] = []

        for idx_, (idx, input, v_est_pre, v_est_last) in tqdm(enumerate(test_dataloader_unscaled)):
            v_est_last = v_est_last.flatten()

            # Generate FDI attack
            z_att_noise, v_att_est_last = case_class.gen_fdi_att_dd(
                z_noise=input, v_est_last=v_est_last,
                ang_no=ang_no, mag_no=mag_no,
                ang_str=ang_str, mag_str=mag_str)
            v_att_est_last = torch.from_numpy(v_att_est_last)

            # Compute attacked P/Q
            pq_att = dd_detector.compute_pq(v_att_est_last)
            pq_norm = (pq_att - dd_detector.f_min) / dd_detector.f_diff
            _, loss_recons = dd_detector.evaluate(pq_norm)

            if loss_recons <= dd_detector.ae_threshold[dd_detector.quantile_idx]:
                TP_DDD[f'({ang_no},{ang_str})'].append(False)
            else:
                TP_DDD[f'({ang_no},{ang_str})'].append(True)

            # Attack strength
            vang_true = np.angle(v_est_last.numpy())
            vang_att = np.angle(v_att_est_last.numpy())
            c_true = (vang_att - vang_true)
            att_strength[f'({ang_no},{ang_str})'].append(np.linalg.norm(c_true, 2))


save_metric(address=f'metric/{sys_config["case_name"]}/metric_ddd_test.npy',
            TP_DDD=TP_DDD,
            att_strength=att_strength,
            FP_DDD=FP_DDD)

In [ ]:
# Load cases, measurement, and load
case_class = load_case()

dd_detector = Evaluation(case_class=case_class)
scaler_ = scaler()

z_noise_summary, v_est_summary = load_measurement()
load_active, load_reactive, pv_active_, pv_reactive_ = load_load_pv()
test_dataloader_scaled, test_dataloader_unscaled, valid_dataloader_scaled, valid_dataloader_unscaled = load_dataset()

metric_ddd = np.load(f'metric/{sys_config["case_name"]}/metric_ddd_test.npy', allow_pickle=True).item()

# Load test P/Q for normal scoring
test_pq_set = BusPQDataset(mode='test')
print(f'Test P/Q samples: {len(test_pq_set)}')

## TPR and FPR

*GraphAE uses per-bus P/Q input and ChebConv settings from `configs/nn_setting.py`.*

In [ ]:
FP_DD_summary = np.zeros(len(dd_detector.quantile))
for i in range(len(test_pq_set)):
    pq, _ = test_pq_set[i]
    _, loss_recons = dd_detector.evaluate(pq)
    for idx_q, quantile in enumerate(dd_detector.quantile):
        if loss_recons >= dd_detector.ae_threshold[idx_q]:
            FP_DD_summary[idx_q] += 1

FPR_DD_summary = FP_DD_summary / len(test_pq_set)
print(f'GraphAE FPR: {FPR_DD_summary}')

In [ ]:
ang_no_list = [1,2,3]
mag_no = 0
ang_str_list = [0.2,0.3]
mag_str = 0

ROC_dict = {}

for ang_no in ang_no_list:
    for ang_str in ang_str_list:

        residual = []
        for _, (idx, input, v_est_pre, v_est_last) in tqdm(enumerate(test_dataloader_unscaled)):
            v_est_last = v_est_last.flatten()

            # Generate attack
            z_att_noise, v_att_est_last = case_class.gen_fdi_att_dd(
                z_noise=input, v_est_last=v_est_last,
                ang_no=ang_no, mag_no=mag_no,
                ang_str=ang_str, mag_str=mag_str)

            # Compute P/Q from attacked voltage, normalize, evaluate
            pq_att = dd_detector.compute_pq(v_att_est_last)
            pq_norm = (pq_att - dd_detector.f_min) / dd_detector.f_diff
            _, loss_recons = dd_detector.evaluate(pq_norm)
            residual.append(loss_recons)

        TPR_DDD_summary = []
        for i, quantile in enumerate(dd_detector.quantile):
            TPR_DDD = np.sum(np.array(residual) >= dd_detector.ae_threshold[i]) / len(residual)
            TPR_DDD_summary.append(TPR_DDD)

        ROC_dict[f'({ang_no},{ang_str})'] = TPR_DDD_summary

In [ ]:
plt.figure()
plt.plot(1-np.array(dd_detector.quantile), FPR_DD_summary, label='Test Set', color=cm.Dark2(0))
plt.plot(1-np.array(dd_detector.quantile), 1-np.array(dd_detector.quantile), label='Valid Set', linestyle='dashed', color=cm.Dark2(0))
plt.vlines(x=1-dd_detector.quantile[dd_detector.quantile_idx], ymin=0, ymax=1, colors='red', linestyles='--', label='Threshold')
plt.xlabel('FPR')
plt.ylabel('FPR')
plt.fill_between(1-np.array(dd_detector.quantile), 1-np.array(dd_detector.quantile), FPR_DD_summary, color=cm.Dark2(0), alpha=0.1)
plt.legend()
plt.ylim([0.0,0.31])
plt.yticks(np.arange(0,0.31,0.05))
plt.xlim([0.0,0.2])
plt.grid()
plt.savefig(f'figures/{sys_config["case_name"]}/false_positive_graphae.pdf')
plt.show()

In [ ]:
plt.figure()
for idx, key in enumerate(ROC_dict.keys()):
    plt.plot(FPR_DD_summary, ROC_dict[key], label=key, color=cm.Dark2(idx), linestyle='-', marker='o')
    if idx == 3:
        break
plt.xlabel('FPR')
plt.ylabel('TPR')
plt.grid()
plt.vlines(x=FPR_DD_summary[dd_detector.quantile_idx], ymin=0, ymax=1, colors='red', linestyles='--', label='Threshold')
plt.legend()
plt.ylim([0.45,1.0])
plt.xlim([0,0.30])
plt.savefig(f'figures/{sys_config["case_name"]}/roc_graphae.pdf')
plt.show()

print(f'Validation set FPR: {1-dd_detector.quantile[dd_detector.quantile_idx]}')
print(f'False positive rate: {np.sum(metric_ddd["FP_DDD"])/len(metric_ddd["FP_DDD"])}')

In [ ]:
TPR_DDD = []
attack_str = []
for key in metric_ddd['TP_DDD'].keys():
    TPR_DDD.append(np.sum(metric_ddd['TP_DDD'][key])/len(metric_ddd['TP_DDD'][key]))
    attack_str.append(metric_ddd['att_strength'][key])
print(TPR_DDD)

plt.plot(TPR_DDD)
plt.title('TPR of GraphAE')
plt.grid()
plt.show()

print(f'Minimum attack strength: {np.min(attack_str,1)}')

In [ ]:
## AUC, F1-score, ACC, FPR, MCC

from sklearn.metrics import auc, precision_score, recall_score, f1_score, accuracy_score, matthews_corrcoef
import numpy as np

try:
    metric_ddd = np.load(f'metric/{sys_config["case_name"]}/metric_ddd_test.npy', allow_pickle=True).item()
except FileNotFoundError:
    print("Warning: metric_ddd_test.npy not found. Run evaluation cells first.")
    metric_ddd = None

if metric_ddd is not None:

    for i in range(0,4):
        try:
            model_key = list(ROC_dict.keys())[i]
            tpr_array = ROC_dict[model_key]
            fpr_array = FPR_DD_summary
            roc_auc = auc(fpr_array, tpr_array)
            print(f"AUC-ROC (Attack Case: {model_key}): {roc_auc:.4f}")
        except NameError:
            print("Warning: ROC_dict or FPR_DD_summary not defined.")
        except IndexError:
            print("Warning: ROC_dict is empty.")

    all_tpr_results = []
    for key in metric_ddd['TP_DDD'].keys():
        all_tpr_results.extend(metric_ddd['TP_DDD'][key])

    TP_FN_count = len(all_tpr_results)
    TP_count = np.sum(all_tpr_results)
    FN_count = TP_FN_count - TP_count

    FP_TN_count = len(metric_ddd['FP_DDD'])
    FP_count = np.sum(metric_ddd['FP_DDD'])
    TN_count = FP_TN_count - FP_count

    Y_true = np.concatenate([np.ones(TP_FN_count), np.zeros(FP_TN_count)])
    all_fp_results = metric_ddd['FP_DDD']
    Y_pred = np.concatenate([np.array(all_tpr_results), np.array(all_fp_results)])

    accuracy = accuracy_score(Y_true, Y_pred)
    precision = precision_score(Y_true, Y_pred, zero_division=0)
    recall = recall_score(Y_true, Y_pred, zero_division=0)
    f1 = f1_score(Y_true, Y_pred, zero_division=0)
    fpr = FP_count / FP_TN_count if FP_TN_count > 0 else 0.0
    mcc = matthews_corrcoef(Y_true, Y_pred)

    print("-" * 45)
    print(f"Metrics for GraphAE (Quantile {dd_detector.quantile[dd_detector.quantile_idx]}):")
    print("-" * 45)
    print(f"  [Sample Counts]")
    print(f"    Total Attacked Samples (TP+FN): {TP_FN_count}")
    print(f"    Total Normal Samples (FP+TN)  : {FP_TN_count}")
    print(f"    True Positives (TP)           : {TP_count}")
    print(f"    False Positives (FP)          : {FP_count}")
    print(f"    True Negatives (TN)           : {TN_count}")
    print(f"    False Negatives (FN)          : {FN_count}")
    print(f"  [Evaluation Metrics]")
    print(f"    Accuracy (ACC)                : {accuracy:.4f}")
    print(f"    Precision                     : {precision:.4f}")
    print(f"    Recall (TPR)                  : {recall:.4f}")
    print(f"    False Positive Rate (FPR)     : {fpr:.4f}")
    print(f"    F1-Score                      : {f1:.4f}")
    print(f"    Matthews Corr Coeff (MCC)     : {mcc:.4f}")
    print("-" * 45)

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

cm = confusion_matrix(Y_true, Y_pred, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()

print("-" * 30)
print("Confusion Matrix (GraphAE):")
print(f" [TN: {tn}]  [FP: {fp}]")
print(f" [FN: {fn}]  [TP: {tp}]")
print("-" * 30)

plt.figure(figsize=(6, 5))
labels = ['Normal (0)', 'Attack (1)']
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels, yticklabels=labels, annot_kws={"size": 16})
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.title('Confusion Matrix: GraphAE', fontsize=14)
plt.show()

if tn == 0 and fp > 0:
    print("\nTN = 0: all normal samples are classified as attacks at the selected threshold.")
if tp == 0 and fn > 0:
    print("\nTP = 0: all attack samples are classified as normal at the selected threshold.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import auc

num_neg = len(test_pq_set)
num_pos = len(test_dataloader_unscaled.dataset)

model_key = list(ROC_dict.keys())[0]
tpr_array = np.array(ROC_dict[model_key])
fpr_array = np.array(FPR_DD_summary)

sort_idx = np.argsort(fpr_array)
fpr_sorted = fpr_array[sort_idx]
tpr_sorted = tpr_array[sort_idx]

roc_auc = auc(fpr_sorted, tpr_sorted)

tp_count = tpr_sorted * num_pos
fp_count = fpr_sorted * num_neg
epsilon = 1e-8

precision_array = tp_count / (tp_count + fp_count + epsilon)
recall_array = tpr_sorted
f1_array = 2 * (precision_array * recall_array) / (precision_array + recall_array + epsilon)

max_f1_idx = np.argmax(f1_array)
max_f1 = f1_array[max_f1_idx]
original_best_idx = sort_idx[max_f1_idx]
best_threshold = dd_detector.ae_threshold[original_best_idx]
best_quantile = dd_detector.quantile[original_best_idx]

print(f"Model Scenario: {model_key}")
print(f"AUC-ROC: {roc_auc:.4f}")
print(f"Max F1-Score: {max_f1:.4f}")
print(f"Best Threshold (from F1): {best_threshold:.4f} (Quantile: {best_quantile})")

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(fpr_sorted, tpr_sorted, label=f'AUC = {roc_auc:.2f}', marker='o')
plt.title('ROC Curve: GraphAE')
plt.xlabel('FPR')
plt.ylabel('TPR')
plt.grid(True)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(fpr_sorted, f1_array, label='F1-Score', color='orange', marker='x')
plt.scatter(fpr_sorted[max_f1_idx], max_f1, color='red', s=100, label='Max F1')
plt.title(f'F1-Score (Max: {max_f1:.2f})')
plt.xlabel('FPR')
plt.ylabel('F1 Score')
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()